# 融合算子：RMSNorm 优化

上一节我们识别了瓶颈：RMSNorm 每层 6 次 kernel 发射，28 层 × 2 = 56 个 RMSNorm，合计 336 次 kernel launch。

本节介绍如何用 NPU 融合算子将 6 次 kernel launch 合并为 1 次，以及如何不修改模型源码完成替换。

---

## 1. RMSNorm vs NPURMSNorm

### 原始 RMSNorm

> 源码：`torchtitan/models/common/rmsnorm.py`

```python
class RMSNorm(nn.RMSNorm, Module):
    @dataclass(kw_only=True, slots=True)
    class Config(Module.Config):
        normalized_shape: int
        eps: float = 1e-5

    def __init__(self, config: Config):
        super().__init__(config.normalized_shape,
                         eps=config.eps)

    def forward(self, x):
        return F.rms_norm(x, self.normalized_shape,
                          self.weight, self.eps)
```

`F.rms_norm` 内部展开为 6 次独立 kernel launch：

```text
x → cast(fp32) → pow(2) → mean → add(eps) → rsqrt → × weight → output
     ①            ②        ③       ④          ⑤        ⑥
```

每次 launch 都要 CPU→NPU 调度 + HBM 读写。56 个 RMSNorm × 6 = **336 次调度**。

### NPURMSNorm（融合后）

通过使用 torch_npu 封装的 Ascend 融合算子，将 6 次 kernel launch 合并为单次 `torch_npu.npu_rms_norm` 调用：

> 源码：`torchtitan_npu/converters/kernels/rms_norm.py`
> <br>API 文档：[`torch_npu.npu_rms_norm`](https://www.hiascend.com/document/detail/zh/Pytorch/730/apiref/torchnpuCustomsapi/docs/context/（beta）torch_npu-npu_rms_norm.md)

```python
class NPURMSNorm(RMSNorm):
    def __init__(self, parent: RMSNorm):
        # Shallow copy of parent's __dict__ is intentional here
        self.__dict__.update(parent.__dict__)
        # ``parent.eps`` may be missing/None on nn.RMSNorm; normalize via _get_eps
        # so forward() sees a clean float (or None to fall back to dtype eps).
        self.eps = _get_eps(parent)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Matches the default implementation of nn.RMSNorm:
        # - Use user-provided eps if it exists.
        # - Otherwise, use the machine epsilon of the current input `x`.
        resolved_eps = self.eps if self.eps is not None else torch.finfo(x.dtype).eps
        return torch_npu.npu_rms_norm(x, self.weight, resolved_eps)[0]
```

对比：

| | 原始 RMSNorm.forward | NPURMSNorm.forward |
|--|---------------------|-------------------|
| 代码行数 | 6 步计算 | 1 行 |
| kernel launch | 6 次 | 1 次 |
| HBM 读写 | 6 读 6 写 | 1 读 1 写 |
| 计算精度 | 取决于 PyTorch | UB 内 fp32，输出转回原 dtype |

`eps` 是防止除零的微小常数。不同库叫法不同（`eps` / `variance_epsilon`）——`_get_eps` 自动兼容。

---

## 2. ModelConverter 机制

> 源码：`torchtitan_npu/converters/kernels/rms_norm.py`

有了 `NPURMSNorm`，如何替换模型里所有旧 RMSNorm？TorchTitan 的 ModelConverter 机制无需修改模型源码：

```python
class NpuRMSNormConverter(ModelCustomConverter):
    def convert(self, model: nn.Module):
        for name, module in model.named_modules():
            if isinstance(module, RMSNorm):                 # ① 找到旧模块
                replace_module_with_name(model, name,       # ② 原地替换
                    NPURMSNorm(module))
```

遍历 → 匹配 → 替换。Qwen3-1.7B 共 56 处（28 层 × 2）。

---

## 3. 注册与配置

将融合算子注册到训练配置中。

> 源码：`torchtitan_npu/converters/kernels/rms_norm.py`

```python
@register_model_converter("npu_rms_norm")      # ← 注册名字
class RMSNormModelConfig(ModelCustomConfig):
    model_converter = NpuRMSNormConverter       # ← 绑定转换器
```


> [!IMPORTANT] 
> 请在训练配置中配置中启用融合算子。

> 源码：`torchtitan_npu/models/qwen3/config_registry.py`

```python
def _qwen3_1_7b_converters() -> ModelConvertersContainer.Config:
    return ModelConvertersContainer.Config(
        converters=[
            get_model_converter_config("npu_rms_norm")
        ],
    )

```

框架启动时按名字找到转换器，自动执行 `convert()`。

---

## 小结

- 原始 RMSNorm.forward → 6 次 kernel launch，336 次调度；NPURMSNorm.forward → 1 行。
- `npu_rms_norm` 在 UB 内 fp32 完成全部计算，HBM 读写从 6+6 降到 1+1。
- `ModelConverter.convert()` 遍历+匹配+替换，不改模型源码。
- `@register_model_converter` 注册名字，配置中一行启用。

下一节，我们将运行 baseline 和 fused 两套配置，用 profiling 验证效果。

## 练习

1. （判断题）原始 RMSNorm 的 `forward` 包含 6 次独立 kernel launch，56 个 RMSNorm 合计 336 次调度，对小模型来说 launch overhead 很大。

2. （判断题）`torch_npu.npu_rms_norm` 将 RMSNorm 的 6 步计算融合为 1 次 kernel，在片上 SRAM（UB）内以 fp32 精度完成。

3. （单选题）`NPURMSNorm.__init__` 使用 `self.__dict__.update(parent.__dict__)` 的目的是什么？
    A. 深拷贝原模块的参数
    B. 复用原模块的 `weight` Parameter（共享内存），对 FSDP 和优化器透明
    C. 重新创建所有参数
    D. 删除原模块

4. （单选题）ModelConverter 的 `convert()` 方法做了什么？
    A. 训练模型
    B. 遍历模型所有子模块，找到 RMSNorm 实例，替换为 NPURMSNorm
    C. 下载 NPU 驱动
    D. 修改模型配置文件

5. （判断题）`@register_model_converter("npu_rms_norm")` 装饰器将名字与转换器绑定，配置中只需一行名字即可启用。

In [ ]:
!cat ./answer/04.02_answer.txt